# Model Risk - Varational Autoencoder (VAE)

In [6]:
import numpy as np
import pandas as pd

from arch import arch_model

import torch
import torch.nn as nn

## 1. Fundamentals

### 1.1 Load data

In [2]:
returns = pd.read_csv("/Users/andre/Documents/modelrisk/data/qrm2025_returns.csv")
returns = returns.iloc[1:].reset_index(drop=True) # reset_index: Setzt den Index neu zurück; 
returns.head()

,STOXX_EU_600,DOWJONES_INDUSTRIALS,MSCI_EM,S&P_U.S._TREASURY_BOND,S&P_GSCI_Commodity
0,0.021227,0.014988,0.015312,0.001415,0.025407
1,-0.001757,-0.001128,0.010706,0.002608,0.000994
2,-0.000513,0.000798,0.006397,-0.002013,0.017457
3,-0.003961,0.003267,-0.007171,-0.000337,-0.009653
4,0.004198,0.001069,0.001962,0.001275,-0.000069


### 1.2 Configuration

In [18]:
WINDOW = 500
# N_FORECAST = len(returns) - WINDOW
N_FORECAST = 30
N_SIM = 10000

ALPHA = 0.05, 0.01 # Signifikanzniveaus für VaR-Berechnung

SCALE = 1000 # scale factor for returns to improve GARCH convergence

SEED = 42

weights = np.ones(len(returns.columns)) / len(returns.columns)

N_FORECAST, weights

(30, array([0.2, 0.2, 0.2, 0.2, 0.2]))

In [7]:
torch.manual_seed(SEED)

#### 1.3 Hyperparameter 

In [8]:
TRADING_DAYS_PER_YEAR = 252

In [ ]:
MAX_EPOCHS     = 200
BATCH_SIZE     = 64

# NN architecture
HIDDEN_DIMS    = (64, 64)
LATENT_DIM     = 3

LEARNING_RATE  = 1e-3
BETA           = 2.0


## 2. Functions for pipeline

### Fit marginals

Helper functions
_-Präfix: signalisiert, dass diese Funtkionen privat sind und nicht direkt von außen aufgerufen werden sollen.

fit_marginals() bleibt die einzig öffentliche Schnittstelle

In [ ]:
def _extract_forecast(res, horizon, scale):
    """
    Extract mu and sigma forecast from fitted arch model.
    """
    import numpy as np

    fc = res.forecast(horizon=horizon)
    return {
        "mu":    fc.mean.iloc[-1, 0] / scale,
        "sigma": np.sqrt(fc.variance.values[-1, 0]) / scale
    }


def _extract_dist_params(params, dist):
    """
    Extract distribution parameters from fitted arch model params.
    """
    return {
        "nu":     params.get("nu")     if dist == "t"     else
                  params.get("eta")    if dist == "skewt" else None,
        "lambda": params.get("lambda") if dist == "skewt" else None
    }

Fit marginals

In [ ]:
def fit_marginals(returns, garch_order=(1, 1), lags=1, dist="t", horizon=1, scale=100):
    """
    Fit an AR(lags)-GARCH(p,q) model to a single return series.

    Parameters
    ----------
    returns : pd.Series
        Raw (unscaled) return series for one asset.
    garch_order : tuple of int, optional
        (p, q) order of the GARCH volatility process. Default is (1, 1).
    lags : int, optional
        Number of autoregressive lags in the mean equation. Default is 1.
    dist : str, optional
        Innovation distribution. One of 'normal', 't', or 'skewt'. Default is 't'.
    horizon : int, optional
        Forecast horizon in periods. Default is 1.
    scale : float, optional
        Multiplicative scaling applied before fitting for numerical stability.
        All forecasts are rescaled back to the original units.

    Returns
    -------
    dict with keys:
        residuals_std : pd.Series   -- standardized residuals; first row is NaN
        mu            : float       -- one-step-ahead mean forecast
        sigma         : float       -- one-step-ahead volatility forecast
        nu            : float|None  -- degrees of freedom, else None
        lambda        : float|None  -- skewness parameter in (-1, 1) for 'skewt', else None
    """
    from arch import arch_model

    p, q = garch_order
    res = arch_model(returns*scale, mean="AR", lags=lags, vol="Garch", p=p, q=q, dist=dist).fit(disp="off", show_warning=False)

    return {
        "residuals_std": res.std_resid,
        **_extract_forecast(res, horizon, scale),
        **_extract_dist_params(res.params, dist)
    }

### Removing NaNs from matrix_residuals_std

In [50]:
def _drop_nan_rows(matrix):
    """
    Drop rows containing any NaN value.
    """
    return matrix[~np.isnan(matrix).any(axis=1)]

Test

In [52]:
test_marginals = fit_marginals(returns["DOWJONES_INDUSTRIALS"])
test_marginals["residuals_std"]

0            NaN
1      -0.174970
2      -0.020122
3       0.327778
4       0.040394
          ...   
2341   -0.099587
2342    2.697244
2343    0.466872
2344   -0.155349
2345    0.470817
Name: std_resid, Length: 2346, dtype: float64

In [57]:
fits = {c: fit_marginals(returns[c], scale=SCALE, dist="skewt") for c in returns.columns}

In [67]:
fits

{'STOXX_EU_600': {'residuals_std': 0            NaN
  1      -0.138269
  2      -0.087065
  3      -0.417156
  4       0.365977
            ...   
  2341   -0.044172
  2342   -0.084076
  2343   -1.677121
  2344    1.978029
  2345    0.326896
  Name: std_resid, Length: 2346, dtype: float64,
  'mu': np.float64(0.0002855272872633573),
  'sigma': np.float64(0.012147514582046636),
  'nu': np.float64(6.0674261825993225),
  'lambda': np.float64(-0.07623465750154602)},
 'DOWJONES_INDUSTRIALS': {'residuals_std': 0            NaN
  1      -0.151081
  2      -0.006982
  3       0.344237
  4       0.059706
            ...   
  2341   -0.104743
  2342    2.737325
  2343    0.489270
  2344   -0.149373
  2345    0.479335
  Name: std_resid, Length: 2346, dtype: float64,
  'mu': np.float64(0.0003062108205816426),
  'sigma': np.float64(0.020444559519060167),
  'nu': np.float64(4.749038853508652),
  'lambda': np.float64(-0.055423401356340475)},
 'MSCI_EM': {'residuals_std': 0            NaN
  1       0.7

In [ ]:
# to_numpy(): konvertiert eine pd.Series in ein reines Numpy-Array
matrix_residuals_std = _drop_nan_rows(np.column_stack([fits[c]["residuals_std"].to_numpy()
                                        for c in returns.columns]))
matrix_residuals_std

array([[-0.13826902, -0.15108112,  0.76024514,  1.35399515,  0.12701187],
       [-0.08706534, -0.00698155,  0.42121575, -0.99734375,  1.41759779],
       [-0.41715624,  0.34423684, -0.89863569, -0.26146784, -0.72333237],
       ...,
       [-1.67712088,  0.48927039,  0.4547514 ,  0.57894364, -0.8521256 ],
       [ 1.97802914, -0.14937262,  1.09396436,  1.86894183,  0.11893243],
       [ 0.32689615,  0.4793347 ,  0.11270887,  0.96841354, -0.07861779]],
      shape=(2345, 5))

## 3. Main pipeline

In [ ]:
def main(returns, weights, window, n_forecast, n_sim, dist="t"):
    """
    Rolling window VaR forecasting pipeline.

    For each t in  0..n_forecast-1:
        1. Take the rolling window returns[t:t+window].
        2. Fit AR(1)-GARCH(1,1)-t per asset -> standardized residuals, mu & sigma forecasts, dist param nu.

    """
    for t in range(n_forecast):

        window_returns = returns.iloc[t:t+WINDOW] # rolling window of returns for model fitting

        # --- 1. Fit marginals ------------------------------------------------------
        fits = {c: fit_marginals(returns[c], scale=SCALE, dist=dist) for c in returns.columns}

        # Extract standardized residuals
        matrix_residuals_std = _drop_nan_rows(np.column_stack([fits[c]["residuals_std"].to_numpy() for c in returns.columns]))

        # Extract forecast values 
        mu = np.column_stack([f["mu"] for f in fits.values()]) # Mean forecast for the day ahead
        sigma = np.column_stack([f["sigma"] for f in fits.values()]) # Volatility forecast for the day ahead

        # Extract dist params
        nu = np.column_stack([f["nu"] for f in fits.values()]) # nu: degrees of freedom for the t-distribution


        # --- 2. VAE refit ---------------------------------------------------













